In [2]:
import matplotlib.pyplot as plt
%matplotlib tk

In [3]:
import numpy as np


def data_model(amplitude_spectrum, custom_xi=None):
    # spits out a realization given a power spectrum
    # power spectrum should have length len(data), i.e. be already distributed
    if custom_xi is None:
        xi = np.random.standard_normal(len(amplitude_spectrum))
    else:
        xi = custom_xi
    return np.fft.ifft(amplitude_spectrum * xi, norm="ortho")


def mean_prior_power_spectrum(k, p):
    # p = params, k fourier modes
    # assumes k[0] = 0 and np ordering of k
    slope = p[0]
    amplitude = p[1]

    tmp = np.abs(k.copy())  # negative modes are just the positive ones mirrored. If you remove abs you will get an error for slope =-1 for example, makes sense.
    tmp[0] = 1  # mask zeromode

    tmp = tmp**slope

    sorter = np.argsort(k)

    tmp = tmp / (np.trapz(tmp[sorter], k[sorter]))
    tmp = amplitude * tmp
    tmp[0] = 1e-30  # fix zeromode

    assert np.all(tmp >=0 )

    return tmp

In [ ]:
s0 = np.sqrt(mean_prior_power_spectrum(k_lengths, (-4, 2e4)))
# sig_amp_spec = np.log(10)
sig_amp_spec = np.exp(-s0)

data_samples = []
for _ in range(10):
    sl = data_model(s0)
    data_samples.append(sl)


amp_spec_samples = []
for _ in range(3):
    # log_s0 = np.log(s0)
    # log_sl = log_s0 + sig_amp_spec * np.random.standard_normal(len(s0))
    # amp_spec_samples.append(np.exp(log_sl))
    sl = s0 + sig_amp_spec * np.random.standard_normal(len(s0))
    amp_spec_samples.append(sl)


fig, axs = plt.subplots(1,2)


for sl in amp_spec_samples:
    axs[0].plot(k_lengths[1:], (sl**2)[1:], "-", markersize=3, alpha=0.1, color="black")

axs[0].plot(k_lengths[1:], (s0**2)[1:], ".", markersize=3, label="Mean prior power spectrum")
axs[0].loglog()

for sl in data_samples:
    axs[1].plot(t, sl, alpha=0.1, color="black")


axs[1].plot(t, data_samples[0], label="Single data realization")
axs[0].legend()
axs[1].legend()
axs[0].set_title("Prior power spectra")
axs[1].set_title("Data realizations from mean prior power spectrum")


